In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from google.colab import drive
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, TrainerCallback, TrainerState, TrainerControl
import psutil
from kaggle_secrets import UserSecretsClient

In [2]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

In [3]:
%%capture
!pip install bitsandbytes
MODEL_NAME = "bigscience/bloom-3b"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config,
    token=hf_token
)

In [4]:
gpu_alloc = torch.cuda.memory_allocated() / 1e9
gpu_cache = torch.cuda.memory_reserved() / 1e9
print(f" GPU alloc: {gpu_alloc:.2f} GB | GPU cache: {gpu_cache:.2f} GB")

 GPU alloc: 1.60 GB | GPU cache: 2.89 GB


In [5]:
%%capture
ds = load_dataset("tatsu-lab/alpaca", token=hf_token)
ds = ds['train'].remove_columns(['instruction', 'input', 'output'])

In [6]:
lora_config = LoraConfig(r=32, target_modules=['query_key_value', 'dense_h_to_4h', 'dense_4h_to_h'])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 34,406,400 || all params: 3,036,963,840 || trainable%: 1.1329


In [7]:
def input_format(example):
  enc = tokenizer(
      example['text'],
      max_length=100,
      padding='max_length',
      truncation=True)
  input_ids=enc['input_ids']
  labels=[-100]*len(input_ids)
  template = tokenizer('### Response:')
  ln = len(template['input_ids'])
  for i in range(len(input_ids)-1):
    if input_ids[i:i+ln] == template['input_ids']:
      labels[i+ln:] = input_ids[i+ln:]
      break
  return {'input_ids':input_ids,
          'attention_mask':enc['attention_mask'],
          'labels':labels}

In [8]:
tokenized_ds = ds.map(input_format, remove_columns=['text'])

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [9]:
split_dataset = tokenized_ds.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [10]:
training_args = TrainingArguments(
    output_dir="./best_model",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=4,
    warmup_steps=200,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    fp16=True
)

In [11]:
class ResourceMonitorCallback(TrainerCallback):
    def on_step_end(self, args, state: TrainerState, control: TrainerControl, **kwargs):
        # Выводим только каждые 10 шагов
        if state.global_step % 10 == 0:
            # GPU
            gpu_alloc = torch.cuda.memory_allocated() / 1e9
            gpu_cache = torch.cuda.memory_reserved() / 1e9

            # CPU / RAM
            cpu_percent = psutil.cpu_percent()
            ram_used = psutil.virtual_memory().used / 1e9
            ram_total = psutil.virtual_memory().total / 1e9

            print(f"Step {state.global_step} | GPU alloc: {gpu_alloc:.2f} GB | GPU cache: {gpu_cache:.2f} GB | CPU: {cpu_percent}% | RAM: {ram_used:.2f}/{ram_total:.2f} GB")

In [12]:
gpu_alloc = torch.cuda.memory_allocated() / 1e9
gpu_cache = torch.cuda.memory_reserved() / 1e9
print(f" GPU alloc: {gpu_alloc:.2f} GB | GPU cache: {gpu_cache:.2f} GB")

 GPU alloc: 1.62 GB | GPU cache: 2.89 GB


In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[ResourceMonitorCallback]
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Epoch,Training Loss,Validation Loss
1,No log,1.564067


Step 10 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 24.6% | RAM: 2.86/33.66 GB
Step 20 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.8% | RAM: 2.85/33.66 GB
Step 30 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 2.85/33.66 GB
Step 40 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.8% | RAM: 2.86/33.66 GB
Step 50 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 2.84/33.66 GB
Step 60 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 2.85/33.66 GB
Step 70 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 2.86/33.66 GB
Step 80 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.8% | RAM: 2.84/33.66 GB
Step 90 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.8% | RAM: 2.83/33.66 GB
Step 100 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.8% | RAM: 2.82/33.66 GB
Step 110 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 2.84/33.66 GB
Step 120 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.8

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step 390 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.8% | RAM: 3.84/33.66 GB
Step 400 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 3.84/33.66 GB
Step 410 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 3.84/33.66 GB
Step 420 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 30.4% | RAM: 3.86/33.66 GB
Step 430 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 3.83/33.66 GB
Step 440 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 3.85/33.66 GB
Step 450 | GPU alloc: 1.67 GB | GPU cache: 14.91 GB | CPU: 25.7% | RAM: 3.83/33.66 GB
